# 07 — When OpenDSS converges to the wrong answer

**Goal:** run the IEEE 4-node feeder through an AI-written pure OpenDSS path and through CEPT, then compare the returned voltage.

**The intentional mistake:** the direct path omits `Set Voltagebases=[12.47 4.16]`. OpenDSS still converges, but the 4.16 kV downstream buses are interpreted on the wrong per-unit base.

**Expected difference:** pure OpenDSS reports about 0.316 pu at Node 4; CEPT carries the declared bus kV values into the solver and reports about 0.948 pu.

In [1]:
#@title 1. Setup — run once
from hashlib import sha256
from urllib.request import urlopen

_bootstrap_url = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/notebooks/_lesson.py"
_bootstrap = urlopen(_bootstrap_url, timeout=60).read()
if sha256(_bootstrap).hexdigest() != "aa5805c306987984ba7ee64d57763db1938cb06052cf80e0f8f26fa84efba30d":
    raise ValueError("Lesson helper hash mismatch")
exec(compile(_bootstrap, "cept-lesson", "exec"), globals())

CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version
cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [1]:
#@title 2. Inputs — one feeder, two execution paths
CASE_PATH = WORKSPACE / "ieee4_node_feeder.json"
CASE_PAYLOAD = ieee4_node_case()
CASE_PATH.write_text(json.dumps(CASE_PAYLOAD, indent=2) + "\n", encoding="utf-8")
table(
    ["declared input", "value", "unit"],
    [("source voltage", 12.47, "kV"), ("downstream voltage", 4.16, "kV"), ("transformer", "T1, 6 MVA", "spec"), ("load", 1800, "kW")],
)


declared input       value       unit
------------------  ----------  --------
source voltage      12.47       kV
downstream voltage   4.16       kV
transformer         T1, 6 MVA   spec
load                1800        kW


In [1]:
#@title 3. Pure OpenDSS — converge on the wrong model
import opendssdirect as dss

# Direct AI-written commands. The intentional omission is Set Voltagebases.
# CalcVoltageBases cannot infer the 4.16 kV downstream base from the
# transformer command alone, so OpenDSS can converge with the wrong per-unit base.
for command in [
    'Clear',
    'New Circuit.ieee4 basekv=12.47 pu=1.0 phases=3 bus1=node1',
    'New Line.line12 bus1=node1.1.2.3 bus2=node2.1.2.3 phases=3 length=0.6096 units=km r1=0.249 x1=0.373 r0=0.536 x0=1.118 c1=0 c0=0',
    'New Transformer.t1 phases=3 windings=2 buses=[node2.1.2.3,node3.1.2.3] kvs=[12.47,4.16] kvas=[6000,6000] conns=[delta,wye] %r=1.0 xhl=6.0',
    'New Line.line34 bus1=node3.1.2.3 bus2=node4.1.2.3 phases=3 length=0.7620 units=km r1=0.249 x1=0.373 r0=0.536 x0=1.118 c1=0 c0=0',
    'New Load.load4 bus1=node4.1.2.3 phases=3 conn=wye kv=4.16 kw=1800 pf=0.9',
    'CalcVoltageBases',
    'Solve',
]:
    dss.Text.Command(command)
pure_converged = bool(dss.Solution.Converged())
pure_iterations = int(dss.Solution.Iterations())
dss.Circuit.SetActiveBus('node4')
pure_load_voltage = float(dss.Bus.puVmagAngle()[0])
pure_loss_kw = float(dss.Circuit.Losses()[0]) / 1000
assert pure_converged
assert pure_load_voltage < 0.50
print('OpenDSS converged, but the 4.16 kV load base was never declared.')


converged: True
iterations: 2
Node 4 voltage A: 0.315818 pu
active loss: 61.3130 kW
OpenDSS converged, but the 4.16 kV load base was never declared.


In [1]:
#@title 4. Pure result — convergence hides the wrong base
table(
    ['pure OpenDSS output', 'value', 'meaning'],
    [
        ('converged', pure_converged, 'solver completed'),
        ('iterations', pure_iterations, 'numeric iteration count'),
        ('Node 4 voltage A', f'{pure_load_voltage:.6f} pu', 'wrong: voltage base was omitted'),
        ('active loss', f'{pure_loss_kw:.4f} kW', 'not the CEPT reference result'),
    ],
)
print('A successful solve is not proof that the model was assembled correctly.')


pure OpenDSS output     value         meaning
-------------------  ------------  --------------------------------
converged            True          solver completed
iterations           2             numeric iteration count
Node 4 voltage A     0.315818 pu   wrong: voltage base omitted
active loss          61.3130 kW    not the CEPT reference result
A successful solve is not proof that the model was assembled correctly.


In [1]:
# The Case-level run remains visible as a normal terminal step.
#@title 5. CEPT — declared kV values reach the solver
!cept case check ieee4_node_feeder.json
!cept study run ieee4_node_feeder.json \
    --out runs/07-pure-vs-cept \
    --force \
    --format text
!cept study verify runs/07-pure-vs-cept --format text


Case OK: ieee4_node_feeder.json
Fingerprint: 38ef70def0c3
Readiness: PASS
Engine: opendss
Study: load_flow

CEPT study result: FINISHED
Result             Finished the balanced load flow and saved the evidence.
Artifacts          runs/07-pure-vs-cept
Status             PASSED

CEPT study check: PASSED
[PASS] Case identity
[PASS] Solver result
[PASS] Saved evidence
Claim              WORKFLOW_VALIDATED


In [1]:
#@title 6. CEPT result — correct voltage, SLD, and plot
RUN_DIR = WORKSPACE / "runs" / "07-pure-vs-cept"
display_sld(RUN_DIR)


Bus,Phase A,Phase B,Phase C,Status
node1,1.0000 pu @ -0.00°,1.0000 pu @ -120.00°,1.0000 pu @ 120.00°,OK
node2,0.9967 pu @ -0.10°,0.9967 pu @ -120.10°,0.9967 pu @ 119.90°,OK
node3,0.9839 pu @ -31.09°,0.9839 pu @ -151.09°,0.9839 pu @ 88.91°,OK
node4,0.9477 pu @ -32.32°,0.9477 pu @ -152.32°,0.9477 pu @ 87.68°,UNDER


In [1]:
#@title 7. Compare the wrong answer with the CEPT result
results = read(RUN_DIR / "results.json")
cept_lf = results["load_flow"]
cept_node4 = next(r["v_pu"] for r in cept_lf["bus_voltages"] if r["bus"] == "node4" and r["phase"] == 1)
table(
    ['same IEEE 4-node feeder', 'pure OpenDSS', 'CEPT'],
    [
        ('declared downstream base', 'omitted', '4.16 kV from Case'),
        ('solver converged', pure_converged, cept_lf["converged"]),
        ('Node 4 voltage A (pu)', f'{pure_load_voltage:.6f}', f'{cept_node4:.6f}'),
        ('active loss (kW)', f'{pure_loss_kw:.4f}', f"{cept_lf['total_loss_kw']:.4f}"),
        ('diagnosis', 'manual inspection required', 'Case kV drives adapter voltage bases'),
        ('saved evidence', 'manual / absent', 'results + SLD + plot + manifest + receipt'),
    ],
)
assert pure_converged and cept_lf["converged"]
assert pure_load_voltage < 0.50 < cept_node4
assert abs(cept_node4 - pure_load_voltage) > 0.50


same IEEE 4-node feeder   pure OpenDSS   CEPT
---------------------  -------------  ------------
declared downstream    omitted        4.16 kV from Case
solver converged      True           True
Node 4 voltage A      0.315818       0.947691
active loss           61.3130        60.5979
diagnosis             manual         Case kV drives bases
saved evidence        manual         results + SLD + receipt


## 9. Interpret — convergence is not correctness

The direct OpenDSS path converged and returned a plausible-looking load-flow result, but the AI omitted `Set Voltagebases=[12.47 4.16]`. OpenDSS therefore used the wrong downstream per-unit base and reported about **0.316 pu** at Node 4.

CEPT starts from the declared Case: Node 3 and Node 4 are explicitly 4.16 kV. The adapter sets the OpenDSS voltage bases from those values before solving, so the same feeder returns about **0.948 pu** and the saved SLD, plot, manifest, and verification receipt describe the run that produced it.

**What CEPT helped with:** the declared engineering input survived the AI-to-solver boundary and the result became reviewable.

**What CEPT did not prove:** the Case is correct for a real project. `cept study verify` validates run identity and evidence; project correctness still needs reviewed source data, operating points, and acceptance criteria.